In [5]:
import cv2
import numpy as np
import math
import os

# Create a directory to save intermediate images
output_dir = "output_steps"
if not os.path.exists(output_dir):
    os.makedirs(output_dir)

def create_test_image():
    """Create a test image with a triangle, square, and circle."""
    img = np.ones((500, 500, 3), dtype=np.uint8) * 255
    cv2.polylines(img, [np.array([[100, 100], [150, 200], [50, 200]])], True, (0, 0, 0), 2)  # Triangle
    cv2.rectangle(img, (200, 100), (300, 200), (0, 0, 0), 2)  # Square
    cv2.circle(img, (400, 150), 50, (0, 0, 0), 2)  # Circle
    test_path = os.path.join(output_dir, "test_shapes.png")
    cv2.imwrite(test_path, img)
    print(f"Created test image at: {test_path}")
    return test_path

def load_and_preprocess_image(image_path):
    # Load image
    img = cv2.imread(image_path)
    if img is None:
        print(f"Image not found at {image_path}! Using test image.")
        image_path = create_test_image()
        img = cv2.imread(image_path)
        if img is None:
            print("Failed to create test image!")
            exit()
    
    # Save original image
    try:
        cv2.imwrite(os.path.join(output_dir, "01_original.jpg"), img)
        print("Saved: 01_original.jpg")
    except Exception as e:
        print(f"Error saving 01_original.jpg: {e}")
    
    # Convert to grayscale
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    try:
        cv2.imwrite(os.path.join(output_dir, "02_grayscale.jpg"), gray)
        print("Saved: 02_grayscale.jpg")
    except Exception as e:
        print(f"Error saving 02_grayscale.jpg: {e}")
    
    # Apply Gaussian blur
    blurred = cv2.GaussianBlur(gray, (5, 5), 1)
    try:
        cv2.imwrite(os.path.join(output_dir, "03_blurred.jpg"), blurred)
        print("Saved: 03_blurred.jpg")
    except Exception as e:
        print(f"Error saving 03_blurred.jpg: {e}")
    
    # Detect edges using Canny with adjusted thresholds
    edges = cv2.Canny(blurred, 30, 100)  # Lowered thresholds for better edge detection
    try:
        cv2.imwrite(os.path.join(output_dir, "04_edges.jpg"), edges)
        print("Saved: 04_edges.jpg")
    except Exception as e:
        print(f"Error saving 04_edges.jpg: {e}")
    
    return img, gray, blurred, edges

def find_contours(edges):
    contours, hierarchy = cv2.findContours(edges, cv2.RETR_TREE, cv2.CHAIN_APPROX_SIMPLE)
    if len(contours) == 0 or hierarchy is None:
        print("No contours or hierarchy found!")
        return []
    
    # Sort contours by area and select up to 5 largest
    contours = sorted(contours, key=cv2.contourArea, reverse=True)[:5]
    print(f"Total Contours Found: {len(contours)}")
    return contours

def draw_contours_on_image(img, contours, output_name, display=True):
    # Create a color copy of the image for drawing contours
    if len(img.shape) == 2:  # Grayscale image
        img_color = cv2.cvtColor(img, cv2.COLOR_GRAY2BGR)
    else:
        img_color = img.copy()
    
    # Draw contours if any exist
    if contours:
        cv2.drawContours(img_color, contours, -1, (0, 255, 0), 2)
    else:
        print(f"No contours to draw for {output_name}")
    
    # Save the image
    try:
        cv2.imwrite(os.path.join(output_dir, output_name), img_color)
        print(f"Saved: {output_name}")
    except Exception as e:
        print(f"Error saving {output_name}: {e}")
    
    # Display the image
    if display:
        cv2.imshow(output_name, img_color)
        cv2.waitKey(0)  # Wait for key press to close window
        cv2.destroyAllWindows()
    
    return img_color

def detect_shape(cnt):
    epsilon = 0.02 * cv2.arcLength(cnt, True)
    approx = cv2.approxPolyDP(cnt, epsilon, True)
    vertices = len(approx)

    M = cv2.moments(cnt)
    if M['m00'] != 0:
        cx = int(M['m10'] / M['m00'])
        cy = int(M['m01'] / M['m00'])
    else:
        cx, cy = 0, 0
    center = (cx, cy)

    area = cv2.contourArea(cnt)

    side_lengths = [
        math.hypot(approx[i][0][0] - approx[(i + 1) % vertices][0][0],
                   approx[i][0][1] - approx[(i + 1) % vertices][0][1])
        for i in range(vertices)
    ]

    angles = []
    for i in range(vertices):
        p0, p1, p2 = approx[i - 1][0], approx[i][0], approx[(i + 1) % vertices][0]
        v1 = (p0[0] - p1[0], p0[1] - p1[1])
        v2 = (p2[0] - p1[0], p2[1] - p1[1])
        dot_prod = v1[0] * v2[0] + v1[1] * v2[1]
        mag1 = math.hypot(*v1)
        mag2 = math.hypot(*v2)
        angle = 0 if mag1 == 0 or mag2 == 0 else math.acos(max(min(dot_prod / (mag1 * mag2), 1), -1)) * 180 / math.pi
        angles.append(angle)

    shape = classify_shape(vertices, side_lengths, angles, cnt, area)
    return {'shape': shape, 'center': center, 'area': area, 'contour': approx, 'sides': side_lengths, 'angles': angles}

def classify_shape(vertices, side_lengths, angles, cnt, area):
    if vertices == 3:
        return "Triangle"
    elif vertices == 4:
        sides_equal = all(abs(side - side_lengths[0]) < 10 for side in side_lengths)
        angles_close_90 = all(abs(angle - 90) < 10 for angle in angles)
        if sides_equal and angles_close_90:
            return "Square"
        elif angles_close_90:
            return "Rectangle"
        else:
            return "Quadrilateral"
    elif vertices > 4:
        circularity = 4 * math.pi * area / (cv2.arcLength(cnt, True) ** 2)
        return "Circle" if circularity > 0.8 else f"{vertices}-gon"
    return "Unknown"

def draw_shapes(img, shape_data_list, display=True):
    for shape_data in shape_data_list:
        cv2.drawContours(img, [shape_data['contour']], -1, (0, 255, 0), 2)
        cv2.putText(img, shape_data['shape'], shape_data['center'], cv2.FONT_HERSHEY_SIMPLEX, 0.5, (255, 0, 0), 1)
        print(f"Detected Shape: {shape_data['shape']} at {shape_data['center']}")
    
    # Save final image with detected shapes
    try:
        cv2.imwrite(os.path.join(output_dir, "06_final_shapes.jpg"), img)
        print("Saved: 06_final_shapes.jpg")
    except Exception as e:
        print(f"Error saving 06_final_shapes.jpg: {e}")
    
    # Display the final image
    if display:
        cv2.imshow('Detected Shapes', img)
        cv2.waitKey(0)
        cv2.destroyAllWindows()

def main():
    # Specify your image path
    image_path = r'C:\Users\ashut\Documents\GitHub\ProjectMajor\IMGTEST\003.png'
    img, gray, blurred, edges = load_and_preprocess_image(image_path)
    
    # Find contours after edge detection
    contours = find_contours(edges)
    
    if len(contours) == 0:
        print("No contours found. Adjust Canny thresholds or use a clearer image.")
        return
    
    # Draw contours on grayscale image
    draw_contours_on_image(gray, contours, "02_grayscale_with_contours.jpg")
    
    # Draw contours on blurred image
    draw_contours_on_image(blurred, contours, "03_blurred_with_contours.jpg")
    
    # Draw contours on edges image
    edges_color = cv2.cvtColor(edges, cv2.COLOR_GRAY2BGR)
    if contours:
        cv2.drawContours(edges_color, contours, -1, (0, 255, 0), 2)
    try:
        cv2.imwrite(os.path.join(output_dir, "04_edges_with_contours.jpg"), edges_color)
        print("Saved: 04_edges_with_contours.jpg")
    except Exception as e:
        print(f"Error saving 04_edges_with_contours.jpg: {e}")
    cv2.imshow("Edges with Contours", edges_color)
    cv2.waitKey(0)
    cv2.destroyAllWindows()
    
    # Detect shapes for all contours
    shape_data_list = [detect_shape(cnt) for cnt in contours]
    
    # Draw and display final shapes with labels
    draw_shapes(img, shape_data_list)

    # Print summary of detected shapes
    print("\nSummary of Detected Shapes:")
    for shape_data in shape_data_list:
        print(f"Shape: {shape_data['shape']} at {shape_data['center']} (Area: {shape_data['area']:.2f})")

if __name__ == "__main__":
    main()

Saved: 01_original.jpg
Saved: 02_grayscale.jpg
Saved: 03_blurred.jpg
Saved: 04_edges.jpg
Total Contours Found: 5
Saved: 02_grayscale_with_contours.jpg
Saved: 03_blurred_with_contours.jpg
Saved: 04_edges_with_contours.jpg
Detected Shape: Rectangle at (366, 79)
Detected Shape: Rectangle at (366, 79)
Detected Shape: 16-gon at (346, 183)
Detected Shape: 12-gon at (514, 182)
Detected Shape: 15-gon at (579, 182)
Saved: 06_final_shapes.jpg

Summary of Detected Shapes:
Shape: Rectangle at (366, 79) (Area: 15486.00)
Shape: Rectangle at (366, 79) (Area: 15480.00)
Shape: 16-gon at (346, 183) (Area: 328.50)
Shape: 12-gon at (514, 182) (Area: 217.50)
Shape: 15-gon at (579, 182) (Area: 167.50)
